# Digital Signals Theory

In [1]:
import numpy as np

## Chapter 8: Fast Fourier transform

### Intuition into the algorithm

* While this chapter is very interesting, the best resource for intuition into the Fast Fourier transform algorithm is the [3. Divide & Conquer: FFT video](https://www.youtube.com/watch?v=iTMn0Kt18tg&t=16s&ab_channel=MITOpenCourseWare) in the lecture series for MIT 6.046J with Prof. Erik Demaine.
* The [Design And Analysis Of Algorithms site at OpenCourseWare@MIT](https://ocw.mit.edu/courses/6-046j-design-and-analysis-of-algorithms-spring-2015/) is also a trove of resources.

### Reference implementation of the radix-2 FFT
* See this answer to a question on Signal Processing @StackExchange: [implementing Prime-factor FFT algorithm](https://dsp.stackexchange.com/questions/78641/implementing-prime-factor-fft-algorithm/78645#78645)

### 8.2 Radix-2 Cooley-Tukey

Assume $N$ is some integral power of 2 such that $N = 2^k$...

#### 8.2.1 Divide and Conquer

We start with Definition 5.2 (page 112), the discrete Fourier transform with Euler's formula:

\begin{flalign}
\qquad X[m]  &= \sum_{n=0}^{N-1} x[n] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot n} & \\\\
\end{flalign}

Divide the above summation into _even indices_ ($n = 0, 2, 4, \ldots$) and _odd indices_ ($n = 1, 3, 5, \ldots$).

Letting $n = 2k$ for the _even indices_ and $n = 2k+1$ for the _odd indices_, we have:

\begin{flalign}
\qquad X[m]  &= \sum_{k=0}^{\frac{N}{2}-1} x[2k] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 2k} + \sum_{k=0}^{\frac{N}{2}-1} x[2k+1] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot (2k +1)} & \\\\
\end{flalign}

Notice the following:

\begin{flalign}
\qquad \frac{2k}{N}  &= \frac{k}{\frac{N}{2}} & \\\\
\end{flalign}


Using this observation, let's first consider only the above summation for the _even indices_:

\begin{flalign}
\qquad X_{even}[m]  &= \sum_{k=0}^{\frac{N}{2}-1} x[2k] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 2k} & \\\\
              &= \sum_{k=0}^{\frac{N}{2}-1} x[2k] \cdot e^{-j \cdot 2\pi \cdot m \cdot \frac{k}{\frac{N}{2}} } &
\end{flalign}

And for the above summation for the _odd indices_, we can do this:

\begin{flalign}
\qquad X_{odd}[m]  &= \sum_{k=0}^{\frac{N}{2}-1} x[2k+1] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot (2k+1)} & \\\\
             &= \sum_{k=0}^{\frac{N}{2}-1} x[2k+1] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot (2k)} \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} } & \\
             &= e^{-j \cdot 2\pi \cdot\frac{m}{N} } \cdot \sum_{k=0}^{\frac{N}{2}-1} x[2k+1] \cdot e^{-j \cdot 2\pi \cdot m \cdot \frac{k}{\frac{N}{2}} } & \\             
\end{flalign}

We can interpret this thusly: the above summation for the _odd indices_ is the DFT $X_{odd}[m]$ of signal $x_{odd} = x[2n+1]$ with a rotation factor of $2\pi \cdot \frac{m}{N}$.

Recall that from Theorem 6.1 (DFT shifting) on page 130, if $x[n]$ is a signal with DFT $X[m]$ and $y[n] = x[n-d]$ (a circular shift of $x$ by $d$ samples), the DFT of $y[n]$ is

\begin{flalign}
\qquad Y[m]  &= X[m] \cdot e^{-j \cdot 2\pi \cdot \frac{m}{N} \cdot d} & \\\\        
\end{flalign}

And so $e^{-j \cdot 2\pi \cdot\frac{m}{N} }$ is exactly the rotation when $d=1$, or when signal $x_{odd}$ _is delayed by one sample!_

Which we can then infer:

\begin{flalign}
\qquad X[m]  &=  X_{even}[m] + e^{-j \cdot 2\pi \cdot\frac{m}{N} } \cdot  X_{odd}[m] & (8.1) \\\\
\end{flalign}

which is great, since that means we can express $X[m]$ of $N$ samples with two smaller DFTs using only the $\frac{N}{2}$ samples!

...

But then how do we calculate _all_ $N$ frequencies if the two smaller DFTs are only across $\frac{N}{2}$ samples? What do we do when $m \geq \frac{N}{2}$?

#### 8.2.2 How many frequencies?

One word: _aliasing._

$X[m]$ is result of calculating similarity between signal $x[n]$ with a sinusoid that completes $m$ cycles across the $N$ samples, or in terms of Hertz:

\begin{flalign}
\qquad f_{m}  &= \frac{m}{N} \cdot f_{s} & \\\\
\end{flalign}


Consider $X_{even}[m]$:
* this DFT involves comparisons to a sinusoid that completes $m$ cycles across $N^{\prime} = \frac{N}{2}$
* the sampling rate for signal $x_{even}[n]$ is $f^{\prime}_{s} = \frac{f{s}}{2}$ as the period between samples is double that of the original signal $x[n]$
* for frequency $m \geq \frac{N}{2}$, Theorem 2.1 (aliasing) (c.f. page 29) tells us that since $f_{a} = \left| f - f_{s} \right|$, this frequency has an alias $f_{a}$ given by:

\begin{flalign}
\qquad f_{a}  &= \frac{m}{N^{\prime}} \cdot f^{\prime}_{s} - f^{\prime}_{s} & \\\\
              &= \frac{m}{\frac{N}{2}} \cdot \frac{f_{s}}{2} - \frac{f_{s}}{2} & \\\\
              &= \frac{m}{N} \cdot f_{s} - \frac{f_{s}}{2} & \\\\
              &= \frac{m}{N} \cdot f_{s} - \frac{\frac{N}{2}}{N} \cdot f_{s} & \\\\
              &= \frac{m - \frac{N}{2}}{N} \cdot f_{s} \\\\
\end{flalign}

* so $X_{even}[m]$ when $m \geq \frac{N}{2}$ can be found via aliasing as $X[m - \frac{N}{2}]$
* for $m \lt \frac{N}{2}$, no aliasing is needed
* to notationally combine both cases where $m \lt \frac{N}{2}$ and $m \geq \frac{N}{2}$, we can use $m \rightarrow \left( m \text{ mod } \frac{N}{2} \right)$

And thus we can tweak / fine-tune (8.1) above into a more perfect form that covers all frequency indices $m \in [0, N-1)$:

\begin{flalign}
\qquad X[m]  &=  X_{even}[m \text{ mod } \frac{N}{2}] + e^{-j \cdot 2\pi \cdot\frac{m}{N} } \cdot  X_{odd}[m \text{ mod } \frac{N}{2}] & (8.1^{\prime}) & \\\\
\end{flalign}


----

### 8.3 Exercises

#### Exercise 8.1

_Optimize the radix-2 method to only compute the positive frequencies..._

First, let's have a look at the radix-2 implementation on pages 161-162.

In [2]:
def fft2(x):
    """
    Compute the DFT of an input x of N = 2**k samples.
    """
    N = len(x)

    if N == 1:
        return x
    else:
        X_even = fft2(x[0::2])
        X_odd  = fft2(x[1::2])

        X = np.zeros(N, dtype=np.complex128)

        for m in range(N):
            m_alias = m % (N//2)
            X[m] = X_even[m_alias] + np.exp(-2j * np.pi * m/N) * X_odd[m_alias]

    return X

##### idiot-check!

In [3]:
N = 16

x = np.random.rand(N) 

assert(np.allclose(fft2(x), np.fft.fft(x)))

<hr width=40%/>

Now let's hack that naive implementation to come up with our own `fft2` that returns only the postive frequencies. This means those up to and including index $\frac{N}{2} + 1$.

In [4]:
def rfft2(x):
    """
    Compute the DFT of an input x of N = 2**k samples for only positive frequencies.
    """
    N = len(x)

    # in other words, we want to calculate frequencies up to N/2 + 1...
    end_idx = (N//2) + 1

    if N == 1:
        return x
    else:
        X_even = fft2(x[0::2])
        X_odd  = fft2(x[1::2])

        X = np.zeros(end_idx, dtype=np.complex128)

        for m in range(end_idx):
            if m < len(X_even):
                # no need for using aliasing in the final index of X
                idx = m
            else:
                # m_alias = m % (N//2), but in this case we know that m % (N//2) == 0
                idx = 0
            X[m] = X_even[idx] + np.exp(-2j * np.pi * m/N) * X_odd[idx]

    return X

##### idiot-check!

In [5]:
x = np.random.rand(N) 

assert(np.allclose(rfft2(x), np.fft.rfft(x)))

----

#### Exercise 8.2

_Given input $N = 3^{k}$ for some $k \in \mathbb{Z}$, modify the radix-2 method to obtain a radix-3 method._

##### Divide and conquer for radix-3 Cooley-Tukey (naive)

To repeat, the DFT in terms of Euler's formula:

\begin{flalign}
\qquad X[m]  &= \sum_{n=0}^{N-1} x[n] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot n} & \\\\
\end{flalign}

Divide the above summation into _three_ groups of indices, $n = 0, 3, 6, \ldots$, $n = 1, 4, 7, \ldots$ and $n = 2, 5, 8, \ldots$ we have:

\begin{flalign}
\qquad X[m]  &= \sum_{k=0}^{\frac{N}{3}-1} x[3k] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 3k} + \sum_{k=0}^{\frac{N}{3}-1} x[3k+1] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot (3k +1)} + \sum_{k=0}^{\frac{N}{3}-1} x[3k+2] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot (3k + 2)} & \\\\
             &= \sum_{k=0}^{\frac{N}{3}-1} x[3k] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 3k} + \left( e^{-j \cdot 2\pi \cdot\frac{m}{N}} \right) \cdot \sum_{k=0}^{\frac{N}{3}-1} x[3k+1] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 3k} + \left( e^{-j \cdot 2\pi \cdot\frac{m}{N}} \right)^{2} \cdot \sum_{k=0}^{\frac{N}{3}-1} x[3k+2] \cdot e^{-j \cdot 2\pi \cdot\frac{m}{N} \cdot 3k} & \\\\
             &= X_{0,3,6\ldots}[m \text{ mod } \frac{N}{3}] + e^{-j \cdot 2\pi \cdot\frac{m}{N} } \cdot  X_{1,4,7\ldots}[m \text{ mod } \frac{N}{3}] + e^{-j \cdot 4\pi \cdot\frac{m}{N} } \cdot  X_{2,5,8\ldots}[m \text{ mod } \frac{N}{3}]& \\\\             
\end{flalign}

##### Let's implement this!

In [6]:
def fft3(x):
    """
    Compute the DFT of an input x of N = 3**k samples.
    """
    N = len(x)

    if N == 1:
        return x
    else:
        X_0 = fft3(x[0::3])
        X_1 = fft3(x[1::3])
        X_2 = fft3(x[2::3])

        X = np.zeros(N, dtype=np.complex128)

        for m in range(N):
            m_alias = m % (N//3)
            X[m] = (
                X_0[m_alias] + 
                np.exp(-2j * np.pi * m/N) * X_1[m_alias] +  
                np.exp(-4j * np.pi * m/N) * X_2[m_alias]
            )

    return X

##### idiot-check!

In [7]:
N = 3**7

x = np.random.rand(N) 

assert(np.allclose(fft3(x), np.fft.fft(x)))